# CUDA Matmul Kernel Test
Open this notebook in Google Colab (Runtime > Change runtime type > GPU)

In [ ]:
!nvidia-smi

In [ ]:
%%writefile matmul.cu
#include <cuda_runtime.h>
#include <stdio.h>

__global__ void matmul_kernel(float *A, float *B, float *C, int M, int K, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < M && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < K; k++) {
            sum += A[row * K + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}

extern "C" {

void cuda_matmul(float *host_A, float *host_B, float *host_C, int M, int K, int N) {
    float *d_A, *d_B, *d_C;

    size_t size_A = M * K * sizeof(float);
    size_t size_B = K * N * sizeof(float);
    size_t size_C = M * N * sizeof(float);

    cudaMalloc(&d_A, size_A);
    cudaMalloc(&d_B, size_B);
    cudaMalloc(&d_C, size_C);

    cudaMemcpy(d_A, host_A, size_A, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, host_B, size_B, cudaMemcpyHostToDevice);

    dim3 threads(16, 16);
    dim3 blocks((N + 15) / 16, (M + 15) / 16);

    matmul_kernel<<<blocks, threads>>>(d_A, d_B, d_C, M, K, N);

    cudaMemcpy(host_C, d_C, size_C, cudaMemcpyDeviceToHost);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
}

}

In [ ]:
!nvcc -shared -Xcompiler -fPIC -o matmul.so matmul.cu
print('Compiled successfully!')

In [ ]:
import ctypes
import numpy as np

lib = ctypes.CDLL('./matmul.so')
lib.cuda_matmul.argtypes = [
    ctypes.POINTER(ctypes.c_float),
    ctypes.POINTER(ctypes.c_float),
    ctypes.POINTER(ctypes.c_float),
    ctypes.c_int,
    ctypes.c_int,
    ctypes.c_int,
]
lib.cuda_matmul.restype = None

def cuda_matmul(a, b):
    a = np.ascontiguousarray(a, dtype=np.float32)
    b = np.ascontiguousarray(b, dtype=np.float32)
    M, K = a.shape
    N = b.shape[1]
    c = np.empty((M, N), dtype=np.float32)
    lib.cuda_matmul(
        a.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
        b.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
        c.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
        M, K, N,
    )
    return c

print('Library loaded!')

In [ ]:
print('=== Test 1: Small matrix (3x4) @ (4x5) ===')
A = np.random.randn(3, 4).astype(np.float32)
B = np.random.randn(4, 5).astype(np.float32)

gpu_result = cuda_matmul(A, B)
cpu_result = A @ B

max_diff = np.max(np.abs(gpu_result - cpu_result))
print(f'GPU result:\n{gpu_result}')
print(f'CPU result:\n{cpu_result}')
print(f'Max difference: {max_diff:.10f}')
print(f'PASS: {max_diff < 1e-5}')

print()
print('=== Test 2: Bigger matrix (128x256) @ (256x64) ===')
A2 = np.random.randn(128, 256).astype(np.float32)
B2 = np.random.randn(256, 64).astype(np.float32)

gpu_result2 = cuda_matmul(A2, B2)
cpu_result2 = A2 @ B2

max_diff2 = np.max(np.abs(gpu_result2 - cpu_result2))
print(f'Max difference: {max_diff2:.10f}')
print(f'PASS: {max_diff2 < 1e-4}')

print()
print('=== Test 3: Non-square (1x1024) @ (1024x1) ===')
A3 = np.random.randn(1, 1024).astype(np.float32)
B3 = np.random.randn(1024, 1).astype(np.float32)

gpu_result3 = cuda_matmul(A3, B3)
cpu_result3 = A3 @ B3

max_diff3 = np.max(np.abs(gpu_result3 - cpu_result3))
print(f'GPU: {gpu_result3.flatten()[0]:.6f}, CPU: {cpu_result3.flatten()[0]:.6f}')
print(f'Max difference: {max_diff3:.10f}')
print(f'PASS: {max_diff3 < 1e-4}')

In [ ]:
import time

sizes = [64, 256, 512, 1024, 2048]

print(f'{"Size":>8} | {"NumPy (CPU)":>12} | {"CUDA (GPU)":>12} | {"Speedup":>8}')
print('-' * 50)

for n in sizes:
    A = np.random.randn(n, n).astype(np.float32)
    B = np.random.randn(n, n).astype(np.float32)

    cuda_matmul(A, B)

    start = time.time()
    for _ in range(10):
        cpu_result = A @ B
    cpu_time = (time.time() - start) / 10

    start = time.time()
    for _ in range(10):
        gpu_result = cuda_matmul(A, B)
    gpu_time = (time.time() - start) / 10

    speedup = cpu_time / gpu_time if gpu_time > 0 else 0
    print(f'{n:>8} | {cpu_time*1000:>9.2f} ms | {gpu_time*1000:>9.2f} ms | {speedup:>7.1f}x')